In [17]:
import pandas as pd
import numpy as np
import os

# Read files
input_path = "./7. FDR Output/final_edges_with_fdr.csv"
df = pd.read_csv(input_path)
total_edges_all = df.shape[0]

# Drop NaN Rows
df = df.dropna(subset=["Pooled FDR"])
#print(df.head())
#total_edges_all = df.shape[0]


# Thresholds
thresholds = [1.0, 0.3, 0.25, 0.2, 0.15, 0.1, 0.05, 0.01]
results = []

for p_thresh in thresholds:
    for fdr_thresh in thresholds:
        # If less than thresholds
        max_p_val = np.nanmax(df[["VECPAC p", "LPS p", "DSS p"]],axis=1)
        is_pval_pass = (max_p_val <= p_thresh)
        
        is_fdr_pass = df["Pooled FDR"] <= fdr_thresh
        matches = df[is_pval_pass & is_fdr_pass]

        # Count total edges
        total_edges = matches.shape[0]

        # Count positive and negative edges
        to_check = ["VECPAC r", "LPS r", "DSS r"]

        #nonnan count is a row with true or false if it is or isn't na
        #so adding them up should be 2 or 3
        non_nan_count = matches[to_check].notna().sum(axis=1)  # Series
        sign_sum = np.nansum(np.sign(matches[to_check].values), axis=1)  # Series
    
        # Element-wise comparisons create boolean masks
        pos_edges = (sign_sum == non_nan_count).sum()  # Count True values
        neg_edges = (sign_sum == -non_nan_count).sum()
    
        #pos_edges = len(df[df["edge_dir"] == 1])
        #neg_edges = len(df[df["edge_dir"] == -1])
    
        if neg_edges > 0:
            pos_neg_ratio = pos_edges/neg_edges 
        else:
            pos_neg_ratio = np.nan


        #  Unique nodes
        unique_nodes = pd.concat([matches["Gene"], matches["Metabolite"]]).unique()
        unique_node_count = len(unique_nodes)
        #print(unique_node_count)


        # Density
        total_edges = matches.shape[0]
        print(total_edges)
        #print(matches)
        if (unique_node_count > 1):
            edges = (unique_node_count * (unique_node_count-1)) / 2
            density = total_edges / edges
        else:
            edges = np.nan
            density = np.nan

        
        # Mean Correlation Coefficient
        #print(matches)
        r_cols = ["VECPAC r", "DSS r", "LPS r"]

        edge_means = matches[r_cols].apply(pd.to_numeric, errors="coerce").mean(axis=1)
        mean_corr = edge_means.mean()
        
        # Append results
        results.append({
            "Individual_Pval": p_thresh,
            "FDR": fdr_thresh,
            "Total_Nodes": unique_node_count,
            "Total_Edges": total_edges,
            "Positive_Edges": pos_edges,
            "Negative_Edges": neg_edges,
            "Pos_to_Neg": pos_neg_ratio,            
            "Density": density,
            "Mean_Corr_Coeff": mean_corr
        })

# Output
out_df = pd.DataFrame(results)
    #columns=["Individual P-value", "FDR Threshold", "Unique Nodes", "Edges", "Positive Edges", "Negative Edges", "Positive:Negative Ratio", "Density", "Mean Correlation Coefficient"]


# Save results
os.makedirs("8. Network Property", exist_ok=True)
out_df.to_csv("./8. Network Property/network_properties_pls_cpx.csv", index=False)

print("Done with QUANTILE")


99046
17782
11893
6558
3221
970
149
32
8877
3701
2844
1889
1128
442
106
27
6637
2881
2247
1525
947
386
93
26
5244
2336
1841
1260
806
332
86
26
2606
1196
950
664
447
197
63
24
1697
751
591
413
298
137
54
24
1038
393
302
207
158
74
35
22
720
237
169
124
94
47
26
22
Done with QUANTILE
